# 🚀 03. Supervised Deep Learning, Hybrid Modeling, and Baselines
### Project: XAI System in Banking: AI Anomaly-Driven Credit Card Fraud Detection and Interpretation
**Objective:** Compare supervised Tabular Deep ResNet (trained with Focal Loss) against the Anomaly-Augmented Hybrid Pipeline and industry-standard Gradient Boosted Decision Trees (XGBoost & LightGBM).

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, torch

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.data.preprocessing import DataPipeline
from src.evaluation.evaluator import ModelEvaluator
from src.utils.device import get_device

device = get_device("auto")
print(f"Device: {device}")

## 1. Focal Loss Formulation
Standard Binary Cross-Entropy struggles with 578:1 imbalance because the millions of easy negatives overwhelm the gradients.
$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$
Where $\gamma = 2.0$ downweights easy examples and $\alpha = 0.25$ balances positive/negative class weights.

In [ ]:
pipeline = DataPipeline(processed_dir=str(ROOT_DIR / "data/processed"))
data = pipeline.load_processed()

# Load Model Comparison Summary Table
summary_path = ROOT_DIR / "results/tables/model_comparison.csv"
if summary_path.exists():
    df_results = pd.read_csv(summary_path)
    print("Trained Benchmark Results on Unseen Test Set:")
    display = df_results[["Model", "AUPRC", "ROC-AUC", "F1-Score", "F2-Score", "Recall", "Precision", "Total Loss ($)"]]
    print(display.to_string(index=False))
else:
    print("Model comparison table not found. Run python scripts/evaluate.py to generate benchmarks.")

## 2. Precision-Recall Curves
Precision-Recall curves are the gold standard for severely imbalanced fraud benchmarks.

In [ ]:
pr_img_path = ROOT_DIR / "results/figures/pr_curves.png"
if pr_img_path.exists():
    from IPython.display import Image
    display(Image(filename=str(pr_img_path)))
else:
    print("PR curve plot not found.")

## 3. Financial Cost Analysis
Financial loss model:
$$\text{Total Financial Cost} = \text{FP} \times \$10 + \text{FN} \times \$500$$
We compare the cost incurred by each architecture.

In [ ]:
if summary_path.exists():
    plt.figure(figsize=(8, 4))
    sns.barplot(data=df_results, x="Model", y="Total Loss ($)", palette="viridis")
    plt.xticks(rotation=25, ha="right")
    plt.title("Expected Banking Financial Loss by Model ($FP=$10, $FN=$500)", fontsize=12, fontweight="bold")
    plt.ylabel("Expected Financial Loss ($)")
    plt.tight_layout()
    plt.show()